In [1]:
# Cell 0: Clone repository and set active working directory
import os

REPO_URL = "https://github.com/muhammadwaqarsaleem/verbascribe-exam.git"
REPO_NAME = "verbascribe-exam"

# If repository is not yet cloned in /content, clone it
if not os.path.exists(REPO_NAME):
    # If the repo is private, use: f"https://{YOUR_GITHUB_TOKEN}@github.com/muhammadwaqarsaleem/verbascribe-exam.git"
    !git clone {REPO_URL}
    %cd {REPO_NAME}
else:
    %cd {REPO_NAME}
    !git pull origin main

# Verify current working directory and dataset presence
print("Active Directory:", os.getcwd())
assert os.path.exists("data/intent_slot_corpus.json"), "Error: data/intent_slot_corpus.json not found!"
print("Repository connected and dataset located successfully.")

Cloning into 'verbascribe-exam'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 52 (delta 21), reused 44 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 52.93 KiB | 950.00 KiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/verbascribe-exam
Active Directory: /content/verbascribe-exam
Repository connected and dataset located successfully.


In [2]:
# Cell 1:

import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split

# 1. Label Encodings
INTENT2ID = {"DICTATION": 0, "EDIT_ACTION": 1, "NAVIGATION": 2}
ID2INTENT = {v: k for k, v in INTENT2ID.items()}

TAG2ID = {"O": 0, "B-TARGET": 1, "I-TARGET": 2, "B-REPLACEMENT": 3, "I-REPLACEMENT": 4}
ID2TAG = {v: k for k, v in TAG2ID.items()}

# 2. Joint Neural Architecture
class JointIntentSlotModel(nn.Module):
    def __init__(self, model_name: str = "distilbert-base-uncased", num_intents: int = 3, num_tags: int = 5):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)

        # Heads
        self.intent_classifier = nn.Linear(hidden_size, num_intents)
        self.slot_classifier = nn.Linear(hidden_size, num_tags)

    def forward(self, input_ids, attention_mask, intent_labels=None, tag_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # (batch, seq_len, hidden)
        pooled_output = sequence_output[:, 0, :]    # [CLS] representation

        pooled_output = self.dropout(pooled_output)
        sequence_output = self.dropout(sequence_output)

        intent_logits = self.intent_classifier(pooled_output)
        slot_logits = self.slot_classifier(sequence_output)

        loss = None
        if intent_labels is not None and tag_labels is not None:
            intent_loss = nn.CrossEntropyLoss()(intent_logits, intent_labels)
            slot_loss = nn.CrossEntropyLoss(ignore_index=-100)(
                slot_logits.view(-1, len(TAG2ID)), tag_labels.view(-1)
            )
            # Weighted joint loss
            loss = intent_loss + 1.5 * slot_loss

        return {"loss": loss, "intent_logits": intent_logits, "slot_logits": slot_logits}

# 3. Dataset Pipeline with Subword Alignment
class DictationDataset(Dataset):
    def __init__(self, data_path: str, tokenizer, max_len: int = 64):
        with open(data_path, "r") as f:
            self.data = json.load(f)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item["tokens"]
        tags = item["bio_tags"]
        intent_id = INTENT2ID[item["intent"]]

        input_ids = [self.tokenizer.cls_token_id]
        tag_ids = [-100]  # Ignore [CLS]

        for word, tag in zip(tokens, tags):
            subwords = self.tokenizer.encode(word, add_special_tokens=False)
            input_ids.extend(subwords)
            # First subword gets the tag, remaining subwords get -100
            tag_ids.extend([TAG2ID[tag]] + [-100] * (len(subwords) - 1))

        input_ids = input_ids[:self.max_len - 1] + [self.tokenizer.sep_token_id]
        tag_ids = tag_ids[:self.max_len - 1] + [-100]

        pad_len = self.max_len - len(input_ids)
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        tag_ids += [-100] * pad_len
        attention_mask = [1 if id != self.tokenizer.pad_token_id else 0 for id in input_ids]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "intent_labels": torch.tensor(intent_id, dtype=torch.long),
            "tag_labels": torch.tensor(tag_ids, dtype=torch.long)
        }

print("Architecture and Dataset ready for training execution. Using torch.optim.AdamW for optimization.")

Architecture and Dataset ready for training execution. Using torch.optim.AdamW for optimization.


In [3]:
!pip install seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=d4ea27c18e5e5f7b1ccf0d4d798353c5ae95998ed015e428574396884619c149
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [4]:
# Cell 2:

import numpy as np
from torch.utils.data import DataLoader, random_split
from seqeval.metrics import classification_report, f1_score as seq_f1_score

# 1. Dataset Instantiation & Train-Val Split
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
full_dataset = DictationDataset("data/intent_slot_corpus.json", tokenizer=tokenizer)

train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = random_split(
    full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16, shuffle=False)

# 2. Model, Optimizer, and Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = JointIntentSlotModel("distilbert-base-uncased", num_intents=len(INTENT2ID), num_tags=len(TAG2ID))
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
epochs = 5

print(f"Training on {device} for {epochs} epochs...")

# 3. Training Loop
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, correct_intents, total_samples = 0.0, 0, 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        intent_labels = batch["intent_labels"].to(device)
        tag_labels = batch["tag_labels"].to(device)

        outputs = model(input_ids, attention_mask, intent_labels, tag_labels)
        loss = outputs["loss"]
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs["intent_logits"], dim=-1)
        correct_intents += (preds == intent_labels).sum().item()
        total_samples += intent_labels.size(0)

    train_acc = correct_intents / total_samples
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch}/{epochs} | Loss: {avg_loss:.4f} | Intent Acc: {train_acc:.4f}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training on cuda for 5 epochs...
Epoch 1/5 | Loss: 0.3784 | Intent Acc: 0.9412
Epoch 2/5 | Loss: 0.0081 | Intent Acc: 1.0000
Epoch 3/5 | Loss: 0.0037 | Intent Acc: 1.0000
Epoch 4/5 | Loss: 0.0023 | Intent Acc: 1.0000
Epoch 5/5 | Loss: 0.0016 | Intent Acc: 1.0000


In [5]:
# Cell 3:
model.eval()
val_correct_intents, val_total = 0, 0
all_true_tags, all_pred_tags = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        intent_labels = batch["intent_labels"].to(device)
        tag_labels = batch["tag_labels"].to(device)

        outputs = model(input_ids, attention_mask)

        # Intent Acc
        preds = torch.argmax(outputs["intent_logits"], dim=-1)
        val_correct_intents += (preds == intent_labels).sum().item()
        val_total += intent_labels.size(0)

        # Slot BIO tags
        slot_preds = torch.argmax(outputs["slot_logits"], dim=-1).cpu().numpy()
        slot_trues = tag_labels.cpu().numpy()

        for p_seq, t_seq in zip(slot_preds, slot_trues):
            true_line, pred_line = [], []
            for p, t in zip(p_seq, t_seq):
                if t != -100:
                    true_line.append(ID2TAG[t])
                    pred_line.append(ID2TAG[p])
            all_true_tags.append(true_line)
            all_pred_tags.append(pred_line)

val_intent_acc = val_correct_intents / val_total
slot_f1 = seq_f1_score(all_true_tags, all_pred_tags)

print("\n--- VALIDATION RESULTS ---")
print(f"Intent Classification Accuracy: {val_intent_acc * 100:.2f}%")
print(f"Slot Filling F1-Score: {slot_f1 * 100:.2f}%")

# Critical Test: Verify Lexical Collision Immunity
test_collision = "the delete operation in red-black trees preserves balance through rotations"
tokens = test_collision.split()
inputs = tokenizer(test_collision, return_tensors="pt").to(device)

with torch.no_grad():
    out = model(inputs["input_ids"], inputs["attention_mask"])
    pred_intent = ID2INTENT[torch.argmax(out["intent_logits"], dim=-1).item()]

print(f"\nCollision Test Sentence: '{test_collision}'")
print(f"Predicted Intent (Expected: DICTATION): {pred_intent}")
assert pred_intent == "DICTATION", "WARNING: Model misclassified lexical collision as a command!"
print("Lexical Collision Test Passed: No false command execution.")


--- VALIDATION RESULTS ---
Intent Classification Accuracy: 100.00%
Slot Filling F1-Score: 100.00%

Collision Test Sentence: 'the delete operation in red-black trees preserves balance through rotations'
Predicted Intent (Expected: DICTATION): DICTATION
Lexical Collision Test Passed: No false command execution.


In [6]:
# Cell 4:
import os

# Save Model Checkpoint
os.makedirs("models/intent_slot_model", exist_ok=True)
torch.save(model.state_dict(), "models/intent_slot_model/pytorch_model.bin")
tokenizer.save_pretrained("models/intent_slot_model")
print("Saved fine-tuned model to models/intent_slot_model/")

# Write production inference class to src/intent_parser.py
intent_parser_code = '''import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from src.schemas import IntentSlotResult

INTENT2ID = {"DICTATION": 0, "EDIT_ACTION": 1, "NAVIGATION": 2}
ID2INTENT = {v: k for k, v in INTENT2ID.items()}
TAG2ID = {"O": 0, "B-TARGET": 1, "I-TARGET": 2, "B-REPLACEMENT": 3, "I-REPLACEMENT": 4}
ID2TAG = {v: k for k, v in TAG2ID.items()}

class JointIntentSlotModel(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_intents=3, num_tags=5):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.intent_classifier = nn.Linear(hidden_size, num_intents)
        self.slot_classifier = nn.Linear(hidden_size, num_tags)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq_out = self.dropout(outputs.last_hidden_state)
        pooled = self.dropout(seq_out[:, 0, :])
        return self.intent_classifier(pooled), self.slot_classifier(seq_out)

class IntentParser:
    def __init__(self, model_dir="models/intent_slot_model"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = JointIntentSlotModel()
        self.model.load_state_dict(torch.load(f"{model_dir}/pytorch_model.bin", map_location=self.device))
        self.model.to(self.device)
        self.model.eval()

    def parse(self, text: str) -> IntentSlotResult:
        inputs = self.tokenizer(text, return_tensors="pt", return_offsets_mapping=True).to(self.device)
        offset_mapping = inputs.pop("offset_mapping")[0].cpu().numpy()

        with torch.no_grad():
            intent_logits, slot_logits = self.model(inputs["input_ids"], inputs["attention_mask"])
            intent_probs = torch.softmax(intent_logits, dim=-1)
            pred_intent_id = torch.argmax(intent_probs, dim=-1).item()
            pred_intent = ID2INTENT[pred_intent_id]
            intent_conf = intent_probs[0, pred_intent_id].item()

            slot_preds = torch.argmax(slot_logits, dim=-1)[0].cpu().numpy()

        target_tokens, repl_tokens = [], []
        words = text.split()

        # Subword alignment to extract target & replacement
        for subword_id, tag_id in enumerate(slot_preds):
            tag = ID2TAG[tag_id]
            start, end = offset_mapping[subword_id]
            if start == end:
                continue
            span = text[start:end]
            if tag in ["B-TARGET", "I-TARGET"]:
                target_tokens.append(span)
            elif tag in ["B-REPLACEMENT", "I-REPLACEMENT"]:
                repl_tokens.append(span)

        target = "".join(target_tokens).strip() or None
        replacement = "".join(repl_tokens).strip() or None

        # Structural action resolution
        action_type = None
        lower_t = text.lower()
        if pred_intent == "EDIT_ACTION":
            if "previous sentence" in lower_t or "last sentence" in lower_t:
                action_type = "DELETE_LAST_SENTENCE"
            elif "previous word" in lower_t or "last word" in lower_t:
                action_type = "DELETE_LAST_WORD"
            elif target and replacement:
                action_type = "SUBSTITUTE_SPAN"
            elif target:
                action_type = "DELETE_SPAN"
            elif "clear" in lower_t or "erase all" in lower_t:
                action_type = "CLEAR_BUFFER"
        elif pred_intent == "NAVIGATION":
            action_type = "READBACK_ALL" if "paragraph" in lower_t or "buffer" in lower_t else "READBACK_LAST_SENTENCE"

        is_collision = any(w in lower_t for w in ["delete", "replace", "clear", "scratch"]) and pred_intent == "DICTATION"

        return IntentSlotResult(
            intent=pred_intent,
            action_type=action_type,
            target_text=target,
            replacement_text=replacement,
            confidence=round(intent_conf, 3),
            is_collision_risk=is_collision
        )
'''

with open("src/intent_parser.py", "w") as f:
    f.write(intent_parser_code)
print("Successfully generated src/intent_parser.py!")

Saved fine-tuned model to models/intent_slot_model/
Successfully generated src/intent_parser.py!


In [12]:
%%bash
# Configure git identity inside the Colab VM
git config --global user.name "muhammadwaqarsaleem"
git config --global user.email "mwaqarsaleem1@gmail.com"

# Stage and commit generated parser
git add src/intent_parser.py
git commit -m "feat(slu): implement production IntentParser with slot-gated token inference"

# Push to GitHub using a GitHub Personal Access Token (PAT)
# Generate token at: GitHub Settings -> Developer settings -> Personal access tokens (classic) with 'repo' scope
GITHUB_TOKEN="<YOUR_GITHUB_TOKEN>"
git push https://${GITHUB_TOKEN}@github.com/muhammadwaqarsaleem/verbascribe-exam.git main

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


To https://github.com/muhammadwaqarsaleem/verbascribe-exam.git
   1facedd..f673306  main -> main


In [9]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
backup_path = "/content/drive/MyDrive/verbascribe_models/intent_slot_model"
shutil.copytree("models/intent_slot_model", backup_path, dirs_exist_ok=True)
print(f"Weights safely backed up to Google Drive at: {backup_path}")

Mounted at /content/drive
Weights safely backed up to Google Drive at: /content/drive/MyDrive/verbascribe_models/intent_slot_model
